# 02 · Isolation Forest case study

Plan steps 4-5. Isolation Forest in two training settings from the taxonomy in Section 2:

- **Setting A, semi-supervised / novelty detection:** fit on **normal training traffic only**, score the test sets.
- **Setting B, unsupervised:** fit on **unlabelled traffic that contains attacks** (how Liu et al. designed
  Isolation Forest). Labels are used only to evaluate.

Why both: 6 of the 21 features are always 0 in normal training traffic, and Isolation Forest can only split
on values that vary in its training data. Setting A therefore never uses them; Setting B can.

Framing rule (CONTEXT.md §5): this *illustrates* the unsupervised approach surveyed in
Section 3. It is not a benchmark and does not claim a best model. Lead with recall/F1, not accuracy.

In [ ]:
# Works both in Google Colab and locally (Jupyter started anywhere inside the repo).
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Sleepycornflake/WebTraffic-Anamoly.git"

if IN_COLAB:
    if not Path("WebTraffic-Anamoly").exists():
        !git clone -q {REPO_URL}
    os.chdir("WebTraffic-Anamoly")
else:
    while not Path("CONTEXT.md").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("Repo root:", ROOT)

In [ ]:
import json, platform
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support,
                             roc_auc_score, average_precision_score, accuracy_score, RocCurveDisplay)
from features import FEATURE_NAMES

SEED = 42
FIG = ROOT / "results/figures"
MET = ROOT / "results/metrics"

import subprocess
if not (ROOT / "data/processed/csic2010_features.csv").exists():   # fresh Colab session: rebuild the data
    for script in ["src/download.py", "src/build_dataset.py"]:
        subprocess.run([sys.executable, script], check=True)
df = pd.read_csv(ROOT / "data/processed/csic2010_features.csv")
train, test = df[df.split == "train"], df[df.split == "test"]
assert (train.label == 0).all(), "training split must be normal-only"
X_train, X_test, y_test = train[FEATURE_NAMES], test[FEATURE_NAMES], test.label.values
print(len(train), "train (normal) /", len(test), "test", np.bincount(y_test))


def evaluate(y, score, pred):
    """score: higher = more anomalous; pred: 1 = flagged as anomalous."""
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
    return {"recall": r, "precision": p, "f1": f1,
            "roc_auc": roc_auc_score(y, score), "average_precision": average_precision_score(y, score),
            "accuracy": accuracy_score(y, pred),
            "confusion_matrix [[TN, FP], [FN, TP]]": confusion_matrix(y, pred).tolist()}


def used_features(m):
    """Features Isolation Forest actually splits on, across all trees."""
    used = {int(f) for est in m.estimators_ for f in est.tree_.feature if f >= 0}
    return [FEATURE_NAMES[i] for i in sorted(used)]

## Setting A: fit on normal training traffic only

### Hyperparameters
Library defaults from the original paper (Liu et al., 2008): 100 trees, sub-sample size 256.
`contamination="auto"` uses the paper's score threshold (0.5) rather than assuming an anomaly rate,
which we could not know in a real deployment. No tuning on the test set.

In [ ]:
PARAMS = dict(n_estimators=100, max_samples=256, contamination="auto",
              max_features=1.0, bootstrap=False, random_state=SEED, n_jobs=-1)
model = IsolationForest(**PARAMS).fit(X_train)

score = -model.score_samples(X_test)          # higher = more anomalous
y_pred = (model.predict(X_test) == -1).astype(int)

In [ ]:
metrics = evaluate(y_test, score, y_pred)
for k, v in metrics.items():
    print(f"{k:38s} {v if isinstance(v, list) else round(v, 4)}")

In [ ]:
# One figure per plot so each can go into the paper on its own.
fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["normal", "anomalous"],
                                        ax=ax, colorbar=False, values_format=",d")
ax.set_title("Confusion matrix (contamination='auto')")
fig.tight_layout(); fig.savefig(FIG / "if_confusion_matrix.png", dpi=200)

fig, ax = plt.subplots(figsize=(5, 4.5))
RocCurveDisplay.from_predictions(y_test, score, ax=ax, name="Isolation Forest")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_title("ROC curve")
fig.tight_layout(); fig.savefig(FIG / "if_roc_curve.png", dpi=200)

fig, ax = plt.subplots(figsize=(6, 4.5))
for lab, name in [(0, "normal"), (1, "anomalous")]:
    ax.hist(score[y_test == lab], bins=60, alpha=0.6, density=True, label=name)
ax.axvline(-model.offset_, color="k", ls="--", lw=1, label="decision threshold")
ax.set_title("Anomaly score by true label"); ax.set_xlabel("anomaly score"); ax.legend()
fig.tight_layout(); fig.savefig(FIG / "if_score_histogram.png", dpi=200)

## Seed sensitivity (cheap robustness check)
Isolation Forest is randomised. Reporting the spread over a few seeds shows the numbers above
are not a lucky draw. This is a stability check, not hyperparameter tuning.

In [ ]:
rows = []
for s in [0, 1, 2, 3, 42]:
    m = IsolationForest(**{**PARAMS, "random_state": s}).fit(X_train)
    pr = (m.predict(X_test) == -1).astype(int)
    _, rr, ff, _ = precision_recall_fscore_support(y_test, pr, average="binary")
    rows.append({"seed": s, "recall": rr, "f1": ff, "roc_auc": roc_auc_score(y_test, -m.score_samples(X_test))})
seeds = pd.DataFrame(rows)
print(seeds.round(4).to_string(index=False))
print(seeds[["recall", "f1", "roc_auc"]].agg(["mean", "std"]).round(4))

## Optional: without the signature-like keyword features
`sql_keyword_count`, `xss_keyword_count`, `path_traversal_flag`, `ssi_cmd_flag`, `crlf_flag` encode
known attack patterns by hand. Re-running without them shows how much the "unsupervised" model leans
on hand-written signatures, which is a point for the limitations / discussion section.

In Setting A all of these except `sql_keyword_count` are always 0 in training, so Isolation Forest never
splits on them: expect this run to differ only through `sql_keyword_count`.

In [ ]:
SIGNATURE = ["sql_keyword_count", "xss_keyword_count", "path_traversal_flag", "ssi_cmd_flag", "crlf_flag"]
generic = [c for c in FEATURE_NAMES if c not in SIGNATURE]
m = IsolationForest(**PARAMS).fit(train[generic])
pr = (m.predict(test[generic]) == -1).astype(int)
_, rr, ff, _ = precision_recall_fscore_support(y_test, pr, average="binary")
ablation = {"features": "generic only", "recall": rr, "f1": ff,
            "roc_auc": roc_auc_score(y_test, -m.score_samples(test[generic]))}
print({k: round(v, 4) if isinstance(v, float) else v for k, v in ablation.items()})

## Setting B: unsupervised (fit on unlabelled traffic that contains attacks)

Same hyperparameters. Isolation Forest assumes anomalies are *few and different*, but the official test
set is 41% anomalous, far from that assumption. So the main unsupervised pool is **all normal test
requests plus a seeded random sample of anomalous ones making up 5% of the pool**. The full test set
(41%) is kept as a second pool to show what happens when the assumption is violated.

The model is fitted and scored on the same pool (there is no separate training data in this setting);
labels are used only for evaluation.

To compare A and B fairly, Setting A's model (fitted on normal training data) is also scored on the 5% pool.
Precision depends on the anomaly rate, so only compare precision between rows evaluated on the same pool;
ROC-AUC does not depend on it.

In [ ]:
UNSUP_RATE = 0.05
normal_te, anom_te = test[test.label == 0], test[test.label == 1]
n_anom = round(UNSUP_RATE / (1 - UNSUP_RATE) * len(normal_te))
pools = {
    "5pct": pd.concat([normal_te, anom_te.sample(n=n_anom, random_state=SEED)]),
    "full_test_41pct": test,
}

settings, models = {}, {"A": model}
for pool_name, pool in pools.items():
    Xp, yp = pool[FEATURE_NAMES], pool.label.values
    # Setting A model, scored on this pool
    settings[f"A_normal_only | pool={pool_name}"] = {
        "n": len(pool), "anomaly_rate": yp.mean(),
        **evaluate(yp, -model.score_samples(Xp), (model.predict(Xp) == -1).astype(int))}
    # Setting B: fit on the unlabelled pool itself
    m = IsolationForest(**PARAMS).fit(Xp)
    models[f"B_{pool_name}"] = m
    settings[f"B_unsupervised | pool={pool_name}"] = {
        "n": len(pool), "anomaly_rate": yp.mean(),
        **evaluate(yp, -m.score_samples(Xp), (m.predict(Xp) == -1).astype(int))}

comparison = pd.DataFrame(settings).T
cols = ["n", "anomaly_rate", "recall", "precision", "f1", "roc_auc", "average_precision", "accuracy"]
comparison[cols].astype(float).round(4)

In [ ]:
# Which features does each model actually use? (the reason Setting B exists)
const = [f for f in FEATURE_NAMES if train[f].nunique() == 1]
print("Always 0 in normal training traffic:", const, "\n")
feature_usage = {}
for name, m in models.items():
    unused = [f for f in FEATURE_NAMES if f not in used_features(m)]
    feature_usage[name] = unused
    print(f"{name:20s} never splits on: {unused}")

In [ ]:
pool = pools["5pct"]
Xp, yp = pool[FEATURE_NAMES], pool.label.values
fig, ax = plt.subplots(figsize=(5, 4.5))
RocCurveDisplay.from_predictions(yp, -model.score_samples(Xp), ax=ax, name="A: normal-only")
RocCurveDisplay.from_predictions(yp, -models["B_5pct"].score_samples(Xp), ax=ax, name="B: unsupervised")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_title("ROC on the 5% pool, both settings")
fig.tight_layout(); fig.savefig(FIG / "if_roc_settings.png", dpi=200)

In [ ]:
# Seed sensitivity for Setting B on the 5% pool (pool sample and model both re-seeded)
rows = []
for s in [0, 1, 2, 3, 42]:
    p = pd.concat([normal_te, anom_te.sample(n=n_anom, random_state=s)])
    m = IsolationForest(**{**PARAMS, "random_state": s}).fit(p[FEATURE_NAMES])
    e = evaluate(p.label.values, -m.score_samples(p[FEATURE_NAMES]), (m.predict(p[FEATURE_NAMES]) == -1).astype(int))
    rows.append({"seed": s, "recall": e["recall"], "f1": e["f1"], "roc_auc": e["roc_auc"]})
seeds_b = pd.DataFrame(rows)
print(seeds_b.round(4).to_string(index=False))
print(seeds_b[["recall", "f1", "roc_auc"]].agg(["mean", "std"]).round(4))

## Save everything needed to reproduce and report

In [ ]:
run = {
    "model": "IsolationForest", "params": PARAMS, "features": FEATURE_NAMES,
    "n_train": len(train), "n_test": len(test), "test_class_counts": np.bincount(y_test).tolist(),
    "metrics": metrics, "seed_sensitivity": seeds.to_dict("records"), "ablation_generic_features": ablation,
    "unsupervised_pool_anomaly_rate": UNSUP_RATE, "settings_comparison": settings,
    "seed_sensitivity_setting_b_5pct": seeds_b.to_dict("records"),
    "features_never_split_on": feature_usage,
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__,
                 "pandas": pd.__version__, "numpy": np.__version__},
}
(MET / "isolation_forest_run.json").write_text(json.dumps(run, indent=2, default=float), encoding="utf-8")
print("Saved", MET / "isolation_forest_run.json")